# Strain stratification

Adapted from the [summer2 documentation](https://summer2.readthedocs.io)
page `examples/08-strain-stratification` at commit
`d1537d6188aba85c33c0449b197eef6ad8b03d6c` of
[monash-emu/summer2](https://github.com/monash-emu/summer2).

Source licence: BSD-2-Clause, Copyright (c) 2022, monash-emu. Prose is carried
and adapted; code is written in summer4 idiom.

summer2 shipped a `StrainStratification` class that built a separate force of
infection per strain. summer4 **rejects that bundled shape** (coverage ledger
`S7` stays `partial` on purpose). The same behaviour is an ordinary `Property`
on `I`/`R` plus one `ForceOfInfection` per strain — conveniently
`ForceOfInfection.per_trait`.

`per_trait` lives on `ForceOfInfection`. Multi-*disease*
models (independent infection processes with distinct infectious compartments)
are a different pattern: multiple infection flows / FOIs on a `FlowModel`, each with its own `infectious=` selector.

For a normal age split, both strata share one $\lambda$. For strains, each
strain $s$ has its own

$$\lambda_s = c_s \, I_s / N.$$


In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd
import plotly.io as pio

from summer4 import (
    Compartments,
    ExitFlow,
    FlowModel,
    Multiply,
    Param,
    Property,
    PropertyData,
    PropertyMap,
    SavePlan,
    SaveRequest,
    TransitionFlow,
)
from summer4.epi import ForceOfInfection, MixingMatrix
from summer4.results.plan import GroupedOutput

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

state = Property("state", ("S", "I", "R"))
strain = Property("strain", ("mild", "wild"))
pop = Property("pop", ("all",))


def plot_comp(res, title):
    return res["comp"].to_pandas().plot(
        title=title, labels={"index": "year", "value": "people"}
    )


def run(model, y0, t0=1990.0, t1=2010.0, params=None):
    plan = SavePlan(
        requests={"comp": SaveRequest(Compartments())},
        ts=jnp.linspace(t0, t1, 101),
    )
    if not isinstance(y0, PropertyData):
        y0 = PropertyData.wrap(model.pmap, jnp.asarray(y0))
    return model.compile().run(
        params or {}, y0, t0=t0, t1=t1, dt=0.1, save=plan, solver="euler"
    )


## Unstratified baseline


In [ ]:
pmap0 = PropertyMap.from_property(state).stratify(pop)
m0 = FlowModel(pmap0)
m0.add_flow(
    TransitionFlow(
        "infection",
        state["S"],
        state["I"],
        ForceOfInfection(
            "infection",
            infectious=state["I"],
            group_by=pop,
            mixing=MixingMatrix(pop, np.array([[1.0]]), check_reciprocal=False),
            kind="frequency",
            contact_rate=2.0,
        ),
    )
)
m0.add_flow(TransitionFlow("recovery", state["I"], state["R"], 1.0 / 3.0))
m0.add_flow(ExitFlow("infection_death", state["I"], 0.05))
y0_base = jnp.zeros(pmap0.size)
y0_base = y0_base.at[pmap0.select(state["S"])].set(990.0)
y0_base = y0_base.at[pmap0.select(state["I"])].set(10.0)
y0_base = PropertyData.wrap(pmap0, y0_base)
res0 = run(m0, y0_base)
assert float(np.max(np.asarray(res0["comp"].select(state["I"]).values.data))) > 10.0
plot_comp(res0, "Unstratified SIR")


## A. Multi-strain via `per_trait`

Stratify `I` and `R` by strain (immunity to one implies immunity to both in
this toy). `ForceOfInfection.per_trait` builds one named FOI per strain, each
with infectious selector `state["I"] & strain[s]`. Destinations are the matching
infected strata.

With equal contact rates and no flow adjusts, the aggregate matches the
unstratified run.


In [ ]:
pmap = (
    PropertyMap.from_property(state)
    .stratify(pop)
    .stratify(strain, where=state["I"] | state["R"])
)

fois = ForceOfInfection.per_trait(
    strain,
    infectious=state["I"],
    group_by=pop,
    mixing=MixingMatrix(pop, np.array([[1.0]]), check_reciprocal=False),
    kind="frequency",
    contact_rate=Param("beta"),
)
assert [f.name for f in fois] == ["infection_mild", "infection_wild"]
assert fois[0].infectious != fois[1].infectious

m = FlowModel(pmap)
for foi in fois:
    trait = foi.name.split("_", 1)[1]
    m.add_flow(
        TransitionFlow(foi.name, state["S"], state["I"] & strain[trait], foi)
    )
m.add_flow(TransitionFlow("recovery", state["I"], state["R"], 1.0 / 3.0))
m.add_flow(ExitFlow("infection_death", state["I"], 0.05))

y0 = jnp.zeros(pmap.size)
y0 = y0.at[pmap.select(state["S"])].set(990.0)
y0 = y0.at[pmap.select(state["I"] & strain["mild"])].set(8.0)
y0 = y0.at[pmap.select(state["I"] & strain["wild"])].set(2.0)
y0 = PropertyData.wrap(pmap, y0)
cm_strain = m.compile()
plan_strain = SavePlan(
    requests={"comp": SaveRequest(Compartments())},
    ts=jnp.linspace(1990.0, 2010.0, 101),
)
res = cm_strain.run(
    {"beta": 2.0}, y0, t0=1990.0, t1=2010.0, dt=0.1, save=plan_strain, solver="euler"
)
i_total = res["comp"].select(state["I"]).total()
assert float(np.max(np.asarray(i_total.values))) > 10.0
plot_comp(res, "Two strains, equal infectivity (80:20 seed)")


## Strain-specific rates

Wild strain: 1.2× death rate and 2× infectivity (contact rate on that FOI).
Each FOI still saves its own $\lambda$ under its own name. Contact rates are
`Param`s so the same compiled model can be reused under `jax.jit`.


In [ ]:
m = FlowModel(pmap)
for trait, pname in (("mild", "beta_mild"), ("wild", "beta_wild")):
    foi = ForceOfInfection(
        f"infection_{trait}",
        infectious=state["I"] & strain[trait],
        group_by=pop,
        mixing=MixingMatrix(pop, np.array([[1.0]]), check_reciprocal=False),
        kind="frequency",
        contact_rate=Param(pname),
    )
    m.add_flow(
        TransitionFlow(foi.name, state["S"], state["I"] & strain[trait], foi)
    )
m.add_flow(TransitionFlow("recovery", state["I"], state["R"], 1.0 / 3.0))
m.add_flow(
    ExitFlow(
        "infection_death",
        state["I"],
        0.05,
        adjust=[Multiply(1.2, where=strain["wild"])],
    )
)

cm_rates = m.compile()
plan = SavePlan(
    requests={
        "comp": SaveRequest(Compartments()),
        "lam_mild": SaveRequest(GroupedOutput("infection_mild")),
        "lam_wild": SaveRequest(GroupedOutput("infection_wild")),
    },
    ts=jnp.linspace(1990.0, 2010.0, 101),
)
params_rates = {"beta_mild": 2.0, "beta_wild": 4.0}
res = cm_rates.run(
    params_rates, y0, t0=1990.0, t1=2010.0, dt=0.1, save=plan, solver="euler"
)

wild_i = np.asarray(res["comp"].select(state["I"] & strain["wild"]).total().values)
mild_i = np.asarray(res["comp"].select(state["I"] & strain["mild"]).total().values)
assert float(np.max(wild_i)) > float(np.asarray(y0.data)[pmap.select(state["I"] & strain["wild"])][0])
assert res["lam_mild"].dims == ("time", "pop")
assert res["lam_wild"].dims == ("time", "pop")
plot_comp(res, "Wild strain more infectious and more lethal")


### Non-interference check (strains)

A model with only the mild FOI matches the joint model when wild's contact rate
is zero — FOIs do not leak (WP6 headline gate).


In [ ]:
def build_joint(beta_mild, beta_wild):
    model = FlowModel(pmap)
    for trait, beta in (("mild", beta_mild), ("wild", beta_wild)):
        foi = ForceOfInfection(
            f"infection_{trait}",
            infectious=state["I"] & strain[trait],
            group_by=pop,
            mixing=MixingMatrix(pop, np.array([[1.0]]), check_reciprocal=False),
            kind="frequency",
            contact_rate=beta,
        )
        model.add_flow(
            TransitionFlow(foi.name, state["S"], state["I"] & strain[trait], foi)
        )
    model.add_flow(TransitionFlow("recovery", state["I"], state["R"], 1.0 / 3.0))
    model.add_flow(ExitFlow("infection_death", state["I"], 0.05))
    return model.compile()


def build_solo_mild(beta_mild):
    model = FlowModel(pmap)
    foi = ForceOfInfection(
        "infection_mild",
        infectious=state["I"] & strain["mild"],
        group_by=pop,
        mixing=MixingMatrix(pop, np.array([[1.0]]), check_reciprocal=False),
        kind="frequency",
        contact_rate=beta_mild,
    )
    model.add_flow(
        TransitionFlow(foi.name, state["S"], state["I"] & strain["mild"], foi)
    )
    model.add_flow(TransitionFlow("recovery", state["I"], state["R"], 1.0 / 3.0))
    model.add_flow(ExitFlow("infection_death", state["I"], 0.05))
    return model.compile()


plan_ni = SavePlan(
    requests={"comp": SaveRequest(Compartments())},
    ts=jnp.linspace(1990.0, 2000.0, 21),
)
solo_mild = build_solo_mild(2.0).run(
    {}, y0, t0=1990.0, t1=2000.0, dt=0.1, save=plan_ni, solver="euler"
)
joint_mild = build_joint(2.0, 0.0).run(
    {}, y0, t0=1990.0, t1=2000.0, dt=0.1, save=plan_ni, solver="euler"
)
np.testing.assert_allclose(
    np.asarray(solo_mild["comp"].select(state["I"] & strain["mild"]).values.data),
    np.asarray(joint_mild["comp"].select(state["I"] & strain["mild"]).values.data),
    rtol=1e-5,
    atol=1e-5,
)
print("non-interference: solo mild matches joint with wild contact_rate=0")


## B. Multi-disease via `FlowModel`

Independent diseases use **separate** infectious compartments (`Ia`, `Ib`) and
two infection flows, each with its own `infectious=`. This is not
`ForceOfInfection.per_trait` — that helper is for splitting one process by a
trait on a shared disease structure. Here each disease has its own FOI because
each infection `TransitionFlow` carries its own `ForceOfInfection`.


In [ ]:
disease = Property("state", ("S", "Ia", "Ib", "R"))
pmap_d = PropertyMap.from_property(disease).stratify(pop)

model = FlowModel(pmap_d)
mixing = MixingMatrix(pop, np.array([[1.0]]), check_reciprocal=False)
model.add_flow(
    TransitionFlow(
        "infection_a",
        disease["S"],
        disease["Ia"],
        ForceOfInfection(
            "infection_a",
            infectious=disease["Ia"],
            group_by=mixing.prop,
            mixing=mixing,
            kind="frequency",
            contact_rate=Param("beta_a"),
        ),
    )
)
model.add_flow(
    TransitionFlow(
        "infection_b",
        disease["S"],
        disease["Ib"],
        ForceOfInfection(
            "infection_b",
            infectious=disease["Ib"],
            group_by=mixing.prop,
            mixing=mixing,
            kind="frequency",
            contact_rate=Param("beta_b"),
        ),
    )
)
model.add_flow(TransitionFlow("recovery_a", disease["Ia"], disease["R"], 1.0 / 3.0))
model.add_flow(TransitionFlow("recovery_b", disease["Ib"], disease["R"], 1.0 / 3.0))

y0_d = jnp.zeros(pmap_d.size)
y0_d = y0_d.at[pmap_d.select(disease["S"])].set(980.0)
y0_d = y0_d.at[pmap_d.select(disease["Ia"])].set(15.0)
y0_d = y0_d.at[pmap_d.select(disease["Ib"])].set(5.0)
y0_d = PropertyData.wrap(pmap_d, y0_d)
cm_disease = model.compile()
plan_d = SavePlan(
    requests={"comp": SaveRequest(Compartments())},
    ts=jnp.linspace(0.0, 40.0, 81),
)
res_d = cm_disease.run(
    {"beta_a": 1.5, "beta_b": 1.2},
    y0_d,
    t0=0.0,
    t1=40.0,
    dt=0.1,
    save=plan_d,
    solver="euler",
)
assert float(np.max(np.asarray(res_d["comp"].select(disease["Ia"]).values.data))) > 15.0
assert float(np.max(np.asarray(res_d["comp"].select(disease["Ib"]).values.data))) > 5.0
plot_comp(res_d, "Two diseases (Ia / Ib), independent FOIs")


### Non-interference check (diseases)

A solo disease-A model matches the joint model when `beta_b=0` on the shared
`Ia` trajectory.


In [ ]:
def build_disease_epi(beta_a, beta_b, *, both: bool):
    m = FlowModel(pmap_d)
    mixing = MixingMatrix(pop, np.array([[1.0]]), check_reciprocal=False)
    m.add_flow(TransitionFlow("infection_a", disease["S"], disease["Ia"], ForceOfInfection("infection_a", infectious=disease["Ia"]
    , group_by=mixing.prop, mixing=mixing, kind="frequency", contact_rate=beta_a)))
    if both:
        m.add_flow(TransitionFlow("infection_b", disease["S"], disease["Ib"], ForceOfInfection("infection_b", infectious=disease["Ib"]
        , group_by=mixing.prop, mixing=mixing, kind="frequency", contact_rate=beta_b)))
        m.add_flow(TransitionFlow("recovery_b", disease["Ib"], disease["R"], 1.0 / 3.0))
    m.add_flow(TransitionFlow("recovery_a", disease["Ia"], disease["R"], 1.0 / 3.0))
    return m.compile()


plan_dni = SavePlan(
    requests={"comp": SaveRequest(Compartments())},
    ts=jnp.linspace(0.0, 20.0, 21),
)
solo_a = build_disease_epi(1.5, 0.0, both=False).run(
    {}, y0_d, t0=0.0, t1=20.0, dt=0.1, save=plan_dni, solver="euler"
)
joint_a = build_disease_epi(1.5, 0.0, both=True).run(
    {}, y0_d, t0=0.0, t1=20.0, dt=0.1, save=plan_dni, solver="euler"
)
np.testing.assert_allclose(
    np.asarray(solo_a["comp"].select(disease["Ia"]).values.data),
    np.asarray(joint_a["comp"].select(disease["Ia"]).values.data),
    rtol=1e-5,
    atol=1e-5,
)
print("non-interference: solo Ia matches joint with beta_b=0")


## Under `jax.jit`

Differentiate through one disease contact rate on the multi-disease
`FlowModel`. (The multi-strain `per_trait` model is equally JIT-safe with a
shared `Param("beta")`.)


In [ ]:
def loss(beta_a):
    res = cm_disease.run(
        {"beta_a": beta_a, "beta_b": jnp.asarray(1.2)},
        y0_d,
        t0=0.0,
        t1=40.0,
        dt=0.1,
        save=plan_d,
        solver="euler",
    )
    return jnp.sum(jnp.asarray(res["comp"].select(disease["Ia"]).values.data))


jitted = jax.jit(loss)
val = jitted(jnp.asarray(1.5))
grad = jax.grad(loss)(jnp.asarray(1.5))
assert jnp.isfinite(val) and jnp.isfinite(grad)
np.testing.assert_allclose(val, loss(jnp.asarray(1.5)), rtol=1e-4)
print(f"disease loss={float(val):.4g}, grad={float(grad):.4g}")


def strain_loss(beta):
    res = cm_strain.run(
        {"beta": beta}, y0, t0=1990.0, t1=2010.0, dt=0.1, save=plan_strain, solver="euler"
    )
    return jnp.sum(jnp.asarray(res["comp"].select(state["I"]).values.data))


jitted_s = jax.jit(strain_loss)
val_s = jitted_s(jnp.asarray(2.0))
grad_s = jax.grad(strain_loss)(jnp.asarray(2.0))
assert jnp.isfinite(val_s) and jnp.isfinite(grad_s)
np.testing.assert_allclose(val_s, strain_loss(jnp.asarray(2.0)), rtol=1e-4)
print(f"strain loss={float(val_s):.4g}, grad={float(grad_s):.4g}")


## Summary

| summer2 | summer4 |
|---|---|
| `StrainStratification` | **Not provided** (`S7` rejected shape) |
| Per-strain FOI | `ForceOfInfection.per_trait` or hand-built FOIs |
| Strain compartments | `pmap.stratify(strain, where=state["I"] \| state["R"])` |
| Multi-disease (exclusive) | Multiple `TransitionFlow` + `ForceOfInfection` with distinct `infectious=` |
| Concurrent multi-disease + comorbidity | Product of disease axes; see {doc}`12-concurrent-diseases` |

Cross-immunity, waning, and strain replacement remain ordinary flows between
compartments — nothing strain-specific beyond the FOI selectors.
